## Install dependencies and spaCy language models

# Install dependencies using requirements.txt in project environment.
# Install spaCy models


## Install torch for compatibility

`torch==1.13.1` and `torchtext==0.14.1`. Use a compatible environment; the project has not been retested against current releases.


In [ ]:
# Library versions can be checked after installing requirements.
import torchtext
print(torchtext.__version__)


## Import  neccessary libraries

In [ ]:
from pathlib import Path
import sys
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from timeit import default_timer as timer

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Launch notebook from project root or notebooks/ directory')
sys.path.insert(0, str(PROJECT_ROOT))
from src.data_preprocessing import (TranslationDataset, load_translation_csv,
    build_text_pipeline, SRC_LANGUAGE, TGT_LANGUAGE, UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX)
from src.transformer_model import (TransformerSeq, PositionalEncoding, TokenEmbedding,
    create_mask, generate_square_subsequent_mask)
from src.train import train_epoch, evaluate
from src.evaluate import translate, greedy_decode, mean_sentence_bleu


## Set Random Seed for Reproducibility

In [ ]:
# Set seed.
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

## Dataset Preparation

In [ ]:
# English source -> French target (set in src/data_preprocessing.py)


In [ ]:
# spaCy tokenizers are initialized by build_text_pipeline(train_dataset) below.


# Dependencies are installed from requirements.txt.


## Stream and sample data from the WMT14 dataset

In [ ]:
# OPTIONAL: download a large WMT14 sample (requires internet and substantial disk space).
# Uncomment to download. Otherwise place your own CSV at DATA_PATH.
DATA_PATH = PROJECT_ROOT / 'data' / 'wmt14_en_fr_10k.csv'
# from datasets import load_dataset
# dataset_stream = load_dataset('wmt14', 'fr-en', split='train', streaming=True)
# sampled_data = []
# for i, sample in enumerate(dataset_stream):
#     sampled_data.append(sample)
#     if i == 500000:
#         break
# DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
# pd.DataFrame(sampled_data).to_csv(DATA_PATH, index=False)


## Load the dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing dataset: {DATA_PATH}. See README for instructions.')
dataset = load_translation_csv(DATA_PATH)
dataset.head()


## Split JSON-like strings into separate columns

In [ ]:
# Dataframe already parsed safely in load_translation_csv().


In [ ]:
dataset.head()

## Split the dataset into training, validation and test sets

In [ ]:
train_csv, test = train_test_split(dataset, test_size=0.1, random_state=42)  # 10% of dataset for testing

In [ ]:

# Step 2: Split train into train and validation
train_csv, valid_csv = train_test_split(train_csv, test_size=0.25, random_state=42)  # 25% of train for validation


In [ ]:
print(len(train_csv))
print(len(valid_csv))
print(len(test))

In [ ]:
train_csv.head()

In [ ]:
valid_csv.head(10)

In [ ]:
print(train_csv['French words/sentences'].iloc[100])
print(train_csv['English words/sentences'].iloc[100])

## Define a custom Dataset class for translation data

In [ ]:
# Custom Dataset class.
class TranslationDataset(Dataset):
  """
  Custom Dataset class for translation data.
  """
  def __init__(self, csv):
    """
    :param csv: Pandas dataframe containing translation data.
    """
    self.csv = csv

  def __len__(self):
    """
    :return: Length of the dataset.
    """
    return len(self.csv)

  def __getitem__(self, idx):
    """
    :param idx: Index of the sample to retrieve.
    :return: Tuple containing English and French translation.
    """
    return(
      self.csv['English words/sentences'].iloc[idx],
      self.csv['French words/sentences'].iloc[idx]
    )

In [ ]:
train_dataset = TranslationDataset(train_csv)
valid_dataset = TranslationDataset(valid_csv)

In [ ]:
iterator = iter(train_dataset)
print(next(iterator))

## Helper function to tokenize dataset samples and build vocabulary for source and target language

In [ ]:
token_transform, vocab_transform, text_transform, collate_fn = build_text_pipeline(train_dataset)


In [ ]:
# Text transforms and collate_fn are built in src/data_preprocessing.py.


## Model and Training Hyperparameters

In [ ]:
SRC_VOCAB_SIZE = len(vocab_transform[SRC_LANGUAGE])
TGT_VOCAB_SIZE = len(vocab_transform[TGT_LANGUAGE])

EMB_SIZE = 128  # Embedding size
NHEAD = 4  # Number of attention heads
FFN_HID_DIM = 512  # Hidden dimension size
BATCH_SIZE = 64
NUM_ENCODER_LAYERS = 2
NUM_DECODER_LAYERS = 2
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # set device
NUM_EPOCHS = 25


## Functions to create masks for Transformer model

In [ ]:
# Defined in src/transformer_model.py and imported above.


## Positional Encoding

In [ ]:
# Defined in src/transformer_model.py and imported above.


## TokenEmbedding

In [ ]:
# Defined in src/transformer_model.py and imported above.


## Transformer module for sequence-to-sequence tasks

In [ ]:
# Defined in src/transformer_model.py and imported above.


In [ ]:
print(torch.cuda.is_available())


## Initialize the Transformer model and move it to the specified device

In [ ]:
model = TransformerSeq(
    NUM_ENCODER_LAYERS,
    NUM_DECODER_LAYERS,
    EMB_SIZE,
    NHEAD,
    SRC_VOCAB_SIZE,
    TGT_VOCAB_SIZE,
    FFN_HID_DIM
).to(DEVICE)

# Total parameters and trainable parameters.
total_params = sum(p.numel() for p in model.parameters())
print(f"{total_params:,} total parameters.")
total_trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad)
print(f"{total_trainable_params:,} training parameters.")
print(model)

## Define the loss function and optimizer

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss(ignore_index=PAD_IDX)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

## Train the model

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)
val_dataloader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)
# train_epoch and evaluate are imported from src/train.py


In [ ]:
train_loss_list, valid_loss_list = [], []
for epoch in range(1, NUM_EPOCHS + 1):
    start_time = timer()
    train_loss = train_epoch(model, optimizer, train_dataloader, loss_fn, DEVICE, PAD_IDX)
    valid_loss = evaluate(model, val_dataloader, loss_fn, DEVICE, PAD_IDX)
    train_loss_list.append(train_loss)
    valid_loss_list.append(valid_loss)
    print(f'Epoch: {epoch}, Train loss: {train_loss:.4f}, Val loss: {valid_loss:.4f}, Time: {timer()-start_time:.1f}s')


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)


## Plots for the train and validation losses

In [ ]:
def save_plots(train_loss, valid_loss):
    plt.figure(figsize=(10, 7))
    plt.plot(train_loss, label='train loss')
    plt.plot(valid_loss, label='validation loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    fig_path = PROJECT_ROOT / 'figures' / 'training_validation_loss.png'
    fig_path.parent.mkdir(exist_ok=True)
    plt.savefig(fig_path, dpi=200, bbox_inches='tight')
    plt.show()


In [ ]:
save_plots(train_loss_list, valid_loss_list)

## Save Model

In [ ]:
torch.save(model.state_dict(), OUTPUT_DIR / 'model_state.pt')


## Inference

In [ ]:
# The trained model remains in memory after training.
# To reload the weights in a new session, first reconstruct the model in cell 46:
# model.load_state_dict(torch.load(OUTPUT_DIR / 'model_state.pt', map_location=DEVICE))
# model.eval()


## Helper function to generate output sequence using a greedy decoding algorithm

In [ ]:
# greedy_decode() and translate() are imported from src/evaluate.py.


## Display translations

## Install the Natural Language Toolkit (NLTK) library

# NLTK is included in requirements.txt.


In [ ]:
# BLEU computation is implemented in src/evaluate.py.


## Split the test dataset

In [ ]:
test1, test2 = train_test_split(test, test_size=0.1, random_state=42) # 10% of the test set

In [ ]:
print(len(test1))
print(len(test2))

## Calculate BLEU scores for translations on the test dataset

In [ ]:
average_bleu = mean_sentence_bleu(model, test2, text_transform, vocab_transform, DEVICE)
print(f'Average sentence BLEU (0-1): {average_bleu:.4f}')
